# OR breakout excursion and return study: 1-, 2-, 5-minute confirmation

**Question:** After a completed candle closes outside the opening range, how far does price extend in the breakout direction? If it returns to the breached OR boundary, how far does it travel through that boundary?

Use the same 30-minute OR (09:30–09:59 ET) and one-minute stock CSVs as the earlier OR study. Independently resample those one-minute bars into 1-, 2-, and 5-minute confirmation candles aligned to 09:30 ET. A signal occurs at the candle's **end** after a close strictly beyond the OR high or low. All subsequent price paths use one-minute bars starting at the signal time; the breakout candle itself is excluded.

The primary sample is the **first breakout of either side per ticker/day/timeframe**. An exploratory all-signals sample records each transition from inside to outside (including re-breaks after a reset); overlapping signals are not independent. Both upper and lower OR boundaries use the same directional formulas. Run an unfiltered inventory, then a separately labeled subset using saved QQQ and ticker gap columns if available. No entry, stop, or profit assumption determines event selection.

At each horizon (15, 30, 60 minutes, session end at 15:55 ET), report the maximum directional extension; if the OR line is touched, time to first touch and maximum move *through* the line after first touch. A return is an OHLC touch, not a close back inside. Also record the first close back inside as a stronger, separate event. If there is no touch, penetration is missing, not zero. Earlier horizons with incomplete minute coverage are excluded rather than silently treated as no return. Session-end windows are labeled incomplete when missing more than 5% of their expected minutes. A touch during the breakout candle is excluded.

This is descriptive path research, not a trade backtest: one-minute OHLC cannot establish within-minute ordering or exact fill/slippage. Percentages are relative to the OR boundary; dollar distances and OR-width multiples are retained. Reports include win/loss/neutral **only in the separate optional trade-outcome extension**, since a breakout path itself has no trade result.

In [ ]:
from google.colab import drive
from pathlib import Path
import pandas as pd, numpy as np, re
from IPython.display import display
drive.mount('/content/drive')
BASE=Path('/content/drive/MyDrive')
ROOT=next((p for p in (BASE/'Backtesting',BASE/'backtesting') if p.is_dir()),None)
if ROOT is None: raise FileNotFoundError('Backtesting folder missing in MyDrive')
OUT=BASE/'OR_Breakout_Excursion_1m_2m_5m'; OUT.mkdir(exist_ok=True)
TZ='America/New_York'
OR_MIN=30
SIGNAL_LAST='15:00'
SESSION_END='15:55'
HORIZONS=(15,30,60,'session_end')
TIMEFRAMES=(1,2,5)
EXCLUDED={'2026-11-04','2026-11-05'}
paths={}
csv_candidates=list(ROOT.rglob('*_1m_*.csv'))
print('Data folder:',ROOT)
print('Candidate one-minute files:',len(csv_candidates))
print('Example paths:',[str(f.relative_to(ROOT)) for f in csv_candidates[:3]])
for f in csv_candidates:
    m=re.match(r'(.+)_1m_[0-9]{4}-[0-9]{2}-[0-9]{2}$',f.stem,re.I)
    if m: paths.setdefault(m.group(1).upper(),[]).append(f)
print('Tickers:',len(paths),'one-minute CSV files:',sum(map(len,paths.values())))
if not paths:
    raise FileNotFoundError('No matching CSVs. Check Data folder and Example paths above; reopen the latest notebook if running a saved Colab copy.')


In [ ]:
def sessions_for_ticker(file_list):
    parts={}
    for f in sorted(file_list):
        try:
            x=pd.read_csv(f)
            tc=next((c for c in x if c.lower() in ('datetime','timestamp','date')),x.columns[0])
            t=pd.to_datetime(x[tc],utc=True,errors='coerce').dt.tz_convert(TZ)
            y=pd.DataFrame({'time':t})
            for c in ('Open','High','Low','Close','Volume'):
                y[c]=pd.to_numeric(x[c],errors='coerce') if c in x else np.nan
            y=y.dropna(subset=['time','Open','High','Low','Close'])
            y=y.set_index('time').between_time('09:30','15:59')
            for date,g in y.groupby(y.index.strftime('%Y-%m-%d')):
                parts.setdefault(date,[]).append(g)
        except Exception as e: print('Skipped',f.name,str(e)[:90])
    out={}
    for date,groups in parts.items():
        d=pd.concat(groups).sort_index()
        d=d[~d.index.duplicated(keep='last')]
        out[date]=d
    return out

def event_candidates(d,date,minutes,high,low):
    start=pd.Timestamp(date+' 09:30',tz=TZ)
    if minutes==1: bars=d[['Open','High','Low','Close']].copy()
    else:
        bars=d[['Open','High','Low','Close']].resample(
            f'{minutes}min',origin=start,label='left',closed='left'
        ).agg({'Open':'first','High':'max','Low':'min','Close':'last'})
        count=d.Close.resample(f'{minutes}min',origin=start,label='left',closed='left').count()
        bars=bars[count.eq(minutes)].dropna()
    prior={'upper':False,'lower':False}
    events=[]
    for stamp,c in bars.iterrows():
        signal=stamp+pd.Timedelta(minutes=minutes)
        if signal<start+pd.Timedelta(minutes=OR_MIN): continue
        if signal>pd.Timestamp(date+' '+SIGNAL_LAST,tz=TZ):break
        upper=bool(c.Close>high);lower=bool(c.Close<low)
        if upper and not prior['upper']:
            events.append(dict(signal_time=signal,side='upper',boundary=high,signal_close=float(c.Close)))
        if lower and not prior['lower']:
            events.append(dict(signal_time=signal,side='lower',boundary=low,signal_close=float(c.Close)))
        prior={'upper':upper,'lower':lower}
    return events

def observations(d,date,e,minutes,horizon,or_width):
    signal=e['signal_time']; end=pd.Timestamp(date+' '+SESSION_END,tz=TZ)
    window_end=min(end,signal+pd.Timedelta(minutes=horizon)) if isinstance(horizon,int) else end
    expected=pd.date_range(signal,window_end,freq='min',inclusive='left')
    x=d.reindex(expected)
    coverage=int(x.Close.notna().sum())
    enough=(coverage==len(expected)) if isinstance(horizon,int) else (
        bool(expected.size) and coverage/len(expected)>=.95)
    if not enough or expected.empty: return None
    # Directional coordinates: positive is beyond the broken boundary; negative is inside.
    s=1 if e['side']=='upper' else -1
    far=s*(x.High-e['boundary']) if s==1 else s*(x.Low-e['boundary'])
    near=s*(x.Low-e['boundary']) if s==1 else s*(x.High-e['boundary'])
    extension=max(0.,float(far.max()))
    touched=near.le(0)
    first=expected[np.flatnonzero(touched.to_numpy())[0]] if touched.any() else pd.NaT
    # Penetration from first touch onward, including the touch minute.
    # Within that minute we cannot know the order of its high and low.
    after=x.loc[first:] if pd.notna(first) else None
    deepest=max(0.,float(-near.loc[first:].min())) if after is not None else np.nan
    # After a return, quantify any *new* outward extension separately.
    post_extend=max(0.,float(far.loc[first:].max())) if after is not None else np.nan
    close_inside=(s*(x.Close-e['boundary'])).lt(0)
    first_close=expected[np.flatnonzero(close_inside.to_numpy())[0]] if close_inside.any() else pd.NaT
    opposite=(x.Low.le(e['boundary']-or_width) if s==1 else x.High.ge(e['boundary']+or_width))
    def pct(v):return 100*v/e['boundary'] if np.isfinite(v) and e['boundary']>0 else np.nan
    return dict(horizon=str(horizon),minutes_available=coverage,minutes_expected=len(expected),
        extension_dollars=extension,extension_pct=pct(extension),extension_or_width=extension/or_width,
        touched=bool(touched.any()),first_touch=first,
        touch_delay_min=(first-signal).total_seconds()/60 if pd.notna(first) else np.nan,
        penetration_dollars=deepest,penetration_pct=pct(deepest),
        penetration_or_width=deepest/or_width if np.isfinite(deepest) else np.nan,
        post_touch_extension_dollars=post_extend,
        closed_inside=bool(close_inside.any()),first_inside_close=first_close,
        opposite_boundary_touched=bool(opposite.any()),
        last_close=float(x.Close.dropna().iloc[-1]),
        last_close_directional_pct=pct(s*(float(x.Close.dropna().iloc[-1])-e['boundary'])))

In [ ]:
records=[]; audit=[]; first_records=[]
for ticker in sorted(paths):
    day_map=sessions_for_ticker(paths[ticker])
    for date,d in day_map.items():
        if date in EXCLUDED: continue
        start=pd.Timestamp(date+' 09:30',tz=TZ)
        opening=d.reindex(pd.date_range(start,periods=OR_MIN,freq='min'))
        if opening[['Open','High','Low','Close']].isna().any().any():
            audit.append(dict(ticker=ticker,date=date,reason='incomplete_30m_OR'));continue
        high=float(opening.High.max());low=float(opening.Low.min());width=high-low
        if width<=0:
            audit.append(dict(ticker=ticker,date=date,reason='zero_OR_width'));continue
        for tf in TIMEFRAMES:
            events=event_candidates(d,date,tf,high,low)
            if not events:
                audit.append(dict(ticker=ticker,date=date,timeframe_min=tf,reason='no_close_breakout'))
            for rank,e in enumerate(events,1):
                base=dict(ticker=ticker,date=date,timeframe_min=tf,event_rank=rank,
                    first_event=(rank==1),or_high=high,or_low=low,or_width=width,
                    side=e['side'],boundary=e['boundary'],signal_close=e['signal_close'],
                    signal_time=e['signal_time'])
                for horizon in HORIZONS:
                    row=observations(d,date,e,tf,horizon,width)
                    if row is None:
                        audit.append(dict(ticker=ticker,date=date,timeframe_min=tf,event_rank=rank,
                                          horizon=str(horizon),reason='insufficient_post_signal_bars'))
                    else: records.append({**base,**row})
    if len(records) and len(records)%200000<10: print('Processed',ticker)
detail=pd.DataFrame(records)
if detail.empty: raise RuntimeError('No complete breakout windows; inspect audit')
detail.to_csv(OUT/'breakout_events_all.csv',index=False)
pd.DataFrame(audit).to_csv(OUT/'coverage_audit.csv',index=False)
print('Event-window rows',len(detail),'ticker-days',detail[['ticker','date']].drop_duplicates().shape[0])
display(detail.head())

In [ ]:
def describe(g):
    ret=g[g.touched]
    return pd.Series({
        'events':len(g),'tickers':g.ticker.nunique(),'dates':g.date.nunique(),
        'returned':int(g.touched.sum()),'return_pct':100*g.touched.mean(),
        'closed_inside':int(g.closed_inside.sum()),
        'closed_inside_pct':100*g.closed_inside.mean(),
        'median_extension_pct':g.extension_pct.median(),
        'p75_extension_pct':g.extension_pct.quantile(.75),
        'p90_extension_pct':g.extension_pct.quantile(.90),
        'median_extension_or_width':g.extension_or_width.median(),
        'median_minutes_to_touch_returners':ret.touch_delay_min.median(),
        'median_penetration_pct_returners':ret.penetration_pct.median(),
        'p75_penetration_pct_returners':ret.penetration_pct.quantile(.75),
        'p90_penetration_pct_returners':ret.penetration_pct.quantile(.90),
        'median_penetration_or_width_returners':ret.penetration_or_width.median(),
        'opposite_boundary_touched':int(g.opposite_boundary_touched.sum())
    })
primary=detail[detail.first_event].copy()
for name,source,keys in [
    ('summary_first_by_side',primary,['timeframe_min','side','horizon']),
    ('summary_first_pooled',primary,['timeframe_min','horizon']),
    ('summary_all_events',detail,['timeframe_min','side','horizon']),
]:
    summary=source.groupby(keys,dropna=False).apply(describe,include_groups=False).reset_index()
    summary.to_csv(OUT/(name+'.csv'),index=False)
    print(name);display(summary.round(3))
# Bins are set before looking at outcomes; report denominators in every bin.
bins=[-1e-12,.1,.25,.5,1,2,np.inf]
labels=['0–0.1%','>0.1–0.25%','>0.25–0.5%','>0.5–1%','>1–2%','>2%']
primary['extension_band']=pd.cut(primary.extension_pct,bins=bins,labels=labels)
bands=primary.groupby(['timeframe_min','side','horizon','extension_band'],
                      observed=True,dropna=False).apply(describe,include_groups=False).reset_index()
bands.to_csv(OUT/'summary_first_by_extension_band.csv',index=False)
display(bands.round(3).head(40))

## Comparisons and interpretation

1. **Same population:** Compare 1-, 2-, and 5-minute results on ticker-days with complete coverage for all three timeframes. Because slower confirmation occurs later, also compare the intersection of actual breakout events by ticker/day/side, while retaining each timeframe's own signal timestamp. The groups are not independent.
2. **Path chronology:** The maximum extension may happen before or after the first return. Use `post_touch_extension_dollars` to distinguish the two. If a return and penetration happen inside one minute, OHLC does not reveal which happened first.
3. **Censoring:** 15/30/60-minute rows require every expected minute; session end tolerates at most 5% missing bars. A no-return result means *no return by that horizon*, not proof it never returns.
4. **Filters:** Only after unfiltered results, join the saved `all_ticker_session_gaps.csv` by ticker/date to compare QQQ gap ±1%, ticker gap groups, and previously excluded dates. Keep filters separate; do not reselect signals after filtering.
5. **Trade question, next phase:** Take fixed signal entries and OR-line limit entries separately; apply actual stops/targets and classify win/loss/neutral using your established 3:55 exit and ≤−0.5% timed-exit loss rule. These path tables alone have no wins or losses.
6. **Stability:** Show month and date summaries, prices and available-history groups; check whether a few ticker-days dominate results. Reserve newly collected weeks for later validation.

In [ ]:
# Paired ticker-day coverage across all three confirmation intervals.
session=primary[primary.horizon.eq('session_end')].copy()
session['key']=session.ticker+'|'+session.date
common=set.intersection(*(set(session.loc[session.timeframe_min.eq(tf),'key']) for tf in TIMEFRAMES))
paired=session[session.key.isin(common)].copy()
paired.to_csv(OUT/'first_events_common_ticker_days.csv',index=False)
print('Common ticker-days with first-event coverage for 1/2/5m:',len(common))
paired_summary=paired.groupby(['timeframe_min','side'],dropna=False).apply(describe,include_groups=False).reset_index()
paired_summary.to_csv(OUT/'paired_common_ticker_days_summary.csv',index=False)
display(paired_summary.round(3))
monthly=primary.assign(month=primary.date.str.slice(0,7)).groupby(
    ['month','timeframe_min','side','horizon'],dropna=False).apply(describe,include_groups=False).reset_index()
monthly.to_csv(OUT/'monthly_first_events.csv',index=False)
print('Saved output folder:',OUT)